# Example M-dwarf Stellar-Parameter Analysis

This notebook demonstrates a small example of the stellar-parameter calculations used in my 2026 Advanced Physics Project at Uppsala University.

It uses **illustrative example measurements rather than the original research catalogue**. The calculation functions are imported from `stellar_parameters.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stellar_parameters import (
    distance_pc,
    distance_error_pc,
    absolute_k,
    absolute_k_error,
    mass_from_mk,
    mass_error,
    radius_from_mk,
    radius_error,
    teff_from_bp_rp,
    teff_error,
    logg,
    logg_error,
)

## 1. Example observations

The table below contains a few illustrative M-dwarf measurements: Gaia parallax and BP−RP colour together with 2MASS K-band photometry.

The values are included only to demonstrate the analysis workflow.

In [ ]:
stars = pd.DataFrame({
    "Name": ["Example A", "Example B", "Example C", "Example D"],
    "parallax_mas": [100.0, 75.0, 50.0, 40.0],
    "parallax_error_mas": [0.10, 0.08, 0.07, 0.06],
    "Kmag": [6.50, 7.40, 8.70, 9.50],
    "Kmag_error": [0.02, 0.02, 0.03, 0.03],
    "bp_rp": [2.0, 2.3, 2.7, 3.0],
    "bp_rp_error": [0.01, 0.01, 0.02, 0.02],
})

stars

## 2. Distance and absolute K-band magnitude

Distance is calculated from the Gaia parallax. The apparent K-band magnitude is then converted to absolute magnitude, $M_K$.

In [ ]:
stars["distance_pc"] = distance_pc(stars["parallax_mas"])
stars["distance_error_pc"] = distance_error_pc(
    stars["parallax_mas"], stars["parallax_error_mas"]
)

stars["Mk"] = absolute_k(stars["Kmag"], stars["parallax_mas"])
stars["Mk_error"] = absolute_k_error(
    stars["Kmag_error"],
    stars["parallax_mas"],
    stars["parallax_error_mas"],
)

stars[["Name", "distance_pc", "distance_error_pc", "Mk", "Mk_error"]]

## 3. Stellar mass and radius

Empirical polynomial relations based on absolute K-band magnitude are used to estimate stellar mass and radius. Formal observational uncertainties are propagated through the relations.

In [ ]:
stars["Mass"] = mass_from_mk(stars["Mk"])
stars["Mass_error"] = mass_error(stars["Mk"], stars["Mk_error"])

stars["Radius"] = radius_from_mk(stars["Mk"])
stars["Radius_error"] = radius_error(stars["Mk"], stars["Mk_error"])

stars[["Name", "Mass", "Mass_error", "Radius", "Radius_error"]]

## 4. Effective temperature

Effective temperature is estimated from Gaia BP−RP colour using the polynomial calibration implemented in `stellar_parameters.py`.

In [ ]:
stars["Teff"] = teff_from_bp_rp(stars["bp_rp"])
stars["Teff_error"] = teff_error(stars["bp_rp"], stars["bp_rp_error"])

stars[["Name", "bp_rp", "Teff", "Teff_error"]]

## 5. Surface gravity

Finally, surface gravity is calculated from the derived mass and radius.

In [ ]:
stars["logg"] = logg(stars["Mass"], stars["Radius"])
stars["logg_error"] = logg_error(
    stars["Mass"],
    stars["Mass_error"],
    stars["Radius"],
    stars["Radius_error"],
)

results = stars[
    ["Name", "distance_pc", "Mk", "Mass", "Radius", "Teff", "logg"]
].copy()

results.round(3)

## 6. Example visualization

A simple mass–radius plot illustrates the derived stellar parameters. Because both mass and radius are estimated from the same $M_K$ calibration input, this relation should not be interpreted as an independent validation of the empirical relations.

In [ ]:
plt.figure(figsize=(6, 4))

plt.errorbar(
    stars["Mass"],
    stars["Radius"],
    xerr=stars["Mass_error"],
    yerr=stars["Radius_error"],
    fmt="o",
    capsize=3,
)

plt.xlabel(r"Mass ($M_\odot$)")
plt.ylabel(r"Radius ($R_\odot$)")
plt.title("Example M-dwarf Mass–Radius Relation")
plt.tight_layout()
plt.show()

## Notes

This notebook is a compact demonstration of the analysis methods used in the research project. The original stellar catalogue is not included.

The propagated uncertainties shown here represent formal measurement uncertainties. Intrinsic scatter in the empirical stellar calibrations is not included.